In [1]:
%pip install -quiet PyPDF2 opensearch-py sentence-transformers

Note: you may need to restart the kernel to use updated packages.



Usage:   
  c:\Users\ksuma\OneDrive\Documents\local-rag-system\.venv\Scripts\python.exe 
-m pip install [options] <requirement specifier> [package-index-options] ...
  c:\Users\ksuma\OneDrive\Documents\local-rag-system\.venv\Scripts\python.exe 
-m pip install [options] -r <requirements file> [package-index-options] ...
  c:\Users\ksuma\OneDrive\Documents\local-rag-system\.venv\Scripts\python.exe 
-m pip install [options] [-e] <vcs project url> ...
  c:\Users\ksuma\OneDrive\Documents\local-rag-system\.venv\Scripts\python.exe 
-m pip install [options] [-e] <local project path> ...
  c:\Users\ksuma\OneDrive\Documents\local-rag-system\.venv\Scripts\python.exe 
-m pip install [options] <archive url/path> ...

no such option: -u


In [2]:
import json
import re
import os
import sys
from typing import Dict, List, Any

from PyPDF2 import PdfReader
from opensearchpy import OpenSearch, helpers
from sentence_transformers import SentenceTransformer

c:\Users\ksuma\OneDrive\Documents\local-rag-system\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
OPENSEARCH_HOST = "localhost"
OPENSEARCH_PORT = 9200
OPENSEARCH_INDEX ="documents"

EMBEDDING_MODEL = " sentence-transformers/all-MiniLM-L6-v2"
EMBEDDING_DIMENSIONS = 384
ASYMETRIC_EMBEDDING = False

TEXT_CHUNK_SIZE = 500
TEXT_CHUNK_OVERLAP = 100


In [20]:
def clean_text(text: str) -> str:
    """Cleans OCR-extracted text by removing unnecessary newlines, hyphens, and correcting common OCR errors.

    Args:
        text (str): The text to clean.

    Returns:
        str: The cleaned text."""

    text = re.sub(r"(\w+)-\n(\w+)", r"\1\2", text)
    text = re.sub(r"(?<!\n)\n(?!\n)", " ", text)
    text = re.sub(r"\n+", "\n", text)
    text = re.sub(r"[ \t]+", " ", text)

    return text.strip()

In [5]:
def chunk_text(text: str, chunk_size: int, overlap: int = 100) -> List[str]:
    """Splits text into chunks with a specified overlap.

    Args:
        text (str): The text to split.
        chunk_size (int): The number of tokens in each chunk.
        overlap (int): The number of tokens to overlap between chunks.

    Returns:
        List[str]: A list of text chunks."""

    text = clean_text(text)
    tokens = text.split(" ")
    chunks = []
    start = 0
    while start <= len(tokens):
        end = start + chunk_size
        chunk_token = tokens[start:end]
        chunk_text = " ".join(chunk_token)
        chunks.append(chunk_text)
        start = end - overlap
    
    return chunks

In [6]:
def get_embedding_model():
    """Loads and returns the sentence transformer embedding model.
    
    Returns:
        SentenceTransformer: The loaded embedding model."""
    model = SentenceTransformer(EMBEDDING_MODEL)
    return model

def generate_embedding(texts: List[str]):
    """
    Generates embeddings for a list of text chunks.
    
    Args:
        texts (List[str]): List of text chunks to embed.
        
    Returns:
        List[numpy.ndarray]: List of embedding vectors.
    """
    model = get_embedding_model()

    if ASYMETRIC_EMBEDDING :
        texts = [f"passage : {text}" for text in texts]

    embedding = model(texts)
    return embedding
    

In [7]:
client = OpenSearch(
    hosts = [{"hosts": OPENSEARCH_HOST, "port": OPENSEARCH_PORT}],
    http_compress = True,
    timeout = 30,
    max_retries = 3,
    retry_on_timeout = True,
)
try:
    info = client.info()
    print(f"Successfully connect to OpenSearch {info['version']['number']}")
except Exception as e:
    print(f"Failed to connect to OpenSearch: {e}")
    print("Make sure OpenSearch is running on localhost:9200")
    raise


Successfully connect to OpenSearch 2.19.2


In [8]:
def create_index_config() -> Dict[str, Any]:
    """
    Creates the index configuration with mappings for text, embeddings, and metadata.
    
    Returns:
        Dict[str, Any]: The index configuration.
    """
    config = {
        "settings": {
            "index": {
                "number_of_shards": 1,
                "number_of_replicas": 0,
                "knn": True,
            }
        },
        "mappings":{
            "properties": {
                "text": {
                    "type": "text"
                },
                "embedding": {
                    "type": "knn_vector",
                    "dimension": EMBEDDING_DIMENSIONS,
                    "method": {
                        "engine":"faiss",
                        "space_type": "l2",
                        "name": "hnsw",
                        "parameters": {}
                    }
                },
                "document_name": {
                    "type": "keyword"
                }
            }
        }
    }
    return config

index_config = create_index_config()
print("\nIndex Configuration:")
print(json.dumps(index_config, indent=2))


Index Configuration:
{
  "settings": {
    "index": {
      "number_of_shards": 1,
      "number_of_replicas": 0,
      "knn": true
    }
  },
  "mappings": {
    "properties": {
      "text": {
        "type": "text"
      },
      "embedding": {
        "type": "knn_vector",
        "dimension": 384,
        "method": {
          "engine": "faiss",
          "space_type": "l2",
          "name": "hnsw",
          "parameters": {}
        }
      },
      "document_name": {
        "type": "keyword"
      }
    }
  }
}


In [9]:
if not client.indices.exists(index=OPENSEARCH_INDEX):
    response = client.indices.create(index=OPENSEARCH_INDEX, body=index_config)
    print(f"\nCreated index {OPENSEARCH_INDEX} with response: {response}")
else:
    print(f"\nIndex {OPENSEARCH_INDEX} already exists")


Index documents already exists


In [14]:
from opensearchpy.exceptions import NotFoundError
pipeline_name = "nlp-search-pipeline"

try:
    result = client.transport.perform_request(
        "GET",
        f"/_search/pipeline/{pipeline_name}"
    )
    print(f"\n Search pipeline {pipeline_name} exists.")
    print(result)
except NotFoundError:
    print(f"\n Search pipeline {pipeline_name} not exists.")
except Exception as e:
    print(f"\n error {e}")


 Search pipeline nlp-search-pipeline exists.
{'nlp-search-pipeline': {'description': 'Post processor for hybrid search', 'phase_results_processors': [{'normalization-processor': {'normalization': {'technique': 'min_max'}, 'combination': {'technique': 'arithmetic_mean', 'parameters': {'weights': [0.3, 0.7]}}}}]}}


In [23]:
pdf_path = "attention is all you need.pdf"

reader = PdfReader(pdf_path)
text = "".join(page.extract_text() for page in reader.pages)
print(f"Extracted {len(text)} characters from {pdf_path}")

print("\nSample of extracted text:")
print(text[:500] + "...")

cleaned_text = clean_text(text)
print(f"\nlen of cleaned text {len(cleaned_text)}")

chunks = chunk_text(cleaned_text, chunk_size=TEXT_CHUNK_SIZE, overlap=TEXT_CHUNK_OVERLAP)
print(f"Split text into {len(chunks)} chunks")

print("\nSample chunk:")
print(chunks[0])

Extracted 39472 characters from attention is all you need.pdf

Sample of extracted text:
Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables and figures in this paper solely for use in journalistic or
scholarly works.
Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.comNoam Shazeer∗
Google Brain
noam@google.comNiki Parmar∗
Google Research
nikip@google.comJakob Uszkoreit∗
Google Research
usz@google.com
Llion Jones∗
Google Research
llion@google.comAidan N. Gomez∗ †
University of Toronto
aidan@cs.toronto.eduŁukasz Kaise...

len of cleaned text 39437
Split text into 15 chunks

Sample chunk:
Provided proper attribution is provided, Google hereby grants permission to reproduce the tables and figures in this paper solely for use in journalistic or scholarly works. Attention Is All You Need Ashish Vaswani∗ Google Brain avaswani@google.comNoam Shazeer∗ Google Brain noam@google.comNiki Parmar∗ Google Research nikip@google.comJako